# In-the-wild (Android) robustness

**Goal.** Run two trained models -- the 12-channel CNN baseline from
`cnn.ipynb` (`models/cnn_best.tflite`) and the 6-channel
orientation-invariant CNN from `06-features-cnn.ipynb`
(`models/cnn_oinv_baseline.keras`) -- against the Android sessions
recorded by the author and compare
per-session and per-class macro-F1.

The hypothesis under test:

Macro-F1 on the Android sessions **rises by *any* amount** for the
6-channel CNN relative to the 12-channel CNN. That is the proof of
concept that orientation-invariant features generalise across
hardware platforms; without it, the architectural changes in the next step
cannot rescue robustness.

Android sensor convention is different from iOS / MotionSense
(different axis directions and units). The 12-channel pipeline already
converts Android → MotionSense convention in
`in_the_wild_testing.ipynb`; we re-use that conversion here. The
6-channel pipeline produces orientation-invariant quantities downstream,
so it is intrinsically insensitive to the unit / sign convention; we
still feed it data that is dimensionally equivalent to the training
data (i.e. user-acceleration expressed in *g*, gravity expressed in *g*).

> Stisen, A. et al. (2015). Smart devices are different: Assessing and mitigating mobile sensing heterogeneities for activity recognition. *SenSys 2015*, 127-140.


In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from ai_edge_litert.interpreter import Interpreter
import keras

_ROOT = Path('..').resolve()
sys.path.insert(0, str(_ROOT))
from utils.orientation_invariant_features import (
    compute_features,
    ORIENTATION_INVARIANT_COLS,
)

ACT_LABELS = ['dws', 'ups', 'wlk', 'jog', 'std', 'sit']
G = 9.80665

labels_df = pd.read_csv('../../data/in_the_wild/labels.csv')
labels_df = labels_df.set_index('session_dir')
print(labels_df[['activity', 'activity_id', 'pocket_orientation']])


            activity  activity_id                 pocket_orientation
session_dir                                                         
dws              dws            0                            natural
ups              ups            1                            natural
hod              wlk            2                            natural
hod2             wlk            2                            natural
hodanje          wlk            2                            natural
hodanje2         wlk            2                            natural
hodanje3         wlk            2                            natural
jog              jog            3                            natural
ED               wlk            2  screen-toward-thigh + upside-down
EG               wlk            2      screen-toward-thigh + upright
KD               wlk            2  camera-toward-thigh + upside-down
KG               wlk            2      camera-toward-thigh + upright


## Section 1 — Load a session into MotionSense schema

For each Android session we load `Orientation.csv`, `Gravity.csv`,
`Gyroscope.csv`, `TotalAcceleration.csv`; resample to 50 Hz on a
20-ms grid; reconstruct `userAcceleration = total − gravity`; convert
units (m/s² → g) and axes (Android → iOS MotionSense convention) using
exactly the mapping from `in_the_wild_testing.ipynb`.

The result is a 12-channel DataFrame with the same column names as the
MotionSense `create_time_series` output, so the same windowing /
preprocessing pipeline can be re-used.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

G = 9.80665

def load_session(session_dir: str) -> pd.DataFrame:
    base = Path('../../data') / session_dir
    df_ori  = pd.read_csv(base / 'Orientation.csv').sort_values('time')
    df_grav = pd.read_csv(base / 'Gravity.csv').sort_values('time')
    df_gyr  = pd.read_csv(base / 'Gyroscope.csv').sort_values('time')
    df_tot  = pd.read_csv(base / 'TotalAcceleration.csv').sort_values('time')

    df = pd.merge_asof(df_ori[['time', 'roll', 'pitch', 'yaw']],
                       df_grav[['time', 'x', 'y', 'z']], on='time',
                       suffixes=('', '_grav'))
    df = pd.merge_asof(df, df_gyr[['time', 'x', 'y', 'z']], on='time',
                       suffixes=('', '_gyro'))
    df = pd.merge_asof(df, df_tot[['time', 'x', 'y', 'z']], on='time',
                       suffixes=('', '_tot_acc'))
                       
    df.columns = [
        'time', 'attitude.roll', 'attitude.pitch', 'attitude.yaw',
        'raw_gravity.x', 'raw_gravity.y', 'raw_gravity.z',
        'rotationRate.x', 'rotationRate.y', 'rotationRate.z',
        'raw_total_acc.x', 'raw_total_acc.y', 'raw_total_acc.z',
    ]

    df['time_dt'] = pd.to_datetime(df['time'])
    df = (df.set_index('time_dt')
            .resample('20ms')
            .mean(numeric_only=True)
            .interpolate(method='linear')
            .reset_index(drop=True))

    
    df['raw_linear_acc.x'] = df['raw_total_acc.x'] - df['raw_gravity.x']
    df['raw_linear_acc.y'] = df['raw_total_acc.y'] - df['raw_gravity.y']
    df['raw_linear_acc.z'] = df['raw_total_acc.z'] - df['raw_gravity.z']

    df['gravity.x'] = -df['raw_gravity.x'] / G
    df['gravity.y'] = -df['raw_gravity.y'] / G
    df['gravity.z'] = -df['raw_gravity.z'] / G

    df['userAcceleration.x'] = -df['raw_linear_acc.x'] / G
    df['userAcceleration.y'] = -df['raw_linear_acc.y'] / G
    df['userAcceleration.z'] = -df['raw_linear_acc.z'] / G

    df['attitude.pitch'] = -df['attitude.pitch']
    
    df['attitude.yaw'] = -(df['attitude.yaw'] - df['attitude.yaw'].iloc[0])

    cols = [
        'attitude.roll', 'attitude.pitch', 'attitude.yaw',
        'gravity.x', 'gravity.y', 'gravity.z',
        'rotationRate.x', 'rotationRate.y', 'rotationRate.z',
        'userAcceleration.x', 'userAcceleration.y', 'userAcceleration.z',
    ]
    
    return df[cols].iloc[150:-150].reset_index(drop=True)

def window_into_batches(arr: np.ndarray, window=128, step=64) -> np.ndarray:
    if len(arr) < window:
        return np.empty((0, window, arr.shape[1]))
    starts = range(0, len(arr) - window + 1, step)
    return np.stack([arr[s:s + window] for s in starts], axis=0)

## Section 2 — Inference: 12-channel CNN (existing TFLite model)

Mixed normalization is identical to `in_the_wild_testing.ipynb`:
channels 0-5 (attitude + gravity) raw, channels 6-11 (rotationRate +
userAcceleration) instance-Z-scored along time.

In [3]:
def normalize_12ch(X: np.ndarray, eps=1e-8) -> np.ndarray:
    out = X.copy().astype(np.float32)
    dyn = out[:, :, 6:12]
    out[:, :, 6:12] = (dyn - dyn.mean(axis=1, keepdims=True)) / (dyn.std(axis=1, keepdims=True) + eps)
    return out


tflite_path = '../../models/cnn_best.tflite'
interp = Interpreter(model_path=tflite_path)
interp.allocate_tensors()
inp_idx = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']
assert tuple(interp.get_input_details()[0]['shape']) == (1, 128, 12)


def predict_12ch(X_norm: np.ndarray) -> np.ndarray:
    out = np.zeros((len(X_norm), 6), dtype=np.float32)
    for i, w in enumerate(X_norm):
        interp.set_tensor(inp_idx, w[None].astype(np.float32))
        interp.invoke()
        out[i] = interp.get_tensor(out_idx)[0]
    return out


## Section 3 — Inference: 6-channel orientation-invariant CNN

Loaded from the Keras checkpoint that notebook 06 produced. Same
mixed-normalization rule used during training of that model: only
`pitch_unwrapped` is left raw, the other five channels are
instance-Z-scored.

In [4]:
oinv_path = '../../models/cnn_oinv_baseline.keras'
oinv_meta_path = '../../models/cnn_oinv_baseline.preproc.json'

cnn_oinv = keras.models.load_model(oinv_path)
with open(oinv_meta_path) as f:
    oinv_meta = json.load(f)
print('oinv preproc:', oinv_meta)

STATIC_IDX = oinv_meta['static_channel_indices']
DYNAMIC_IDX = oinv_meta['dynamic_channel_indices']


def normalize_oinv(X: np.ndarray, eps=1e-8) -> np.ndarray:
    out = X.copy().astype(np.float32)
    dyn = out[:, :, DYNAMIC_IDX]
    out[:, :, DYNAMIC_IDX] = (dyn - dyn.mean(axis=1, keepdims=True)) / (dyn.std(axis=1, keepdims=True) + eps)
    return out


def predict_oinv(X_norm: np.ndarray) -> np.ndarray:
    return cnn_oinv.predict(X_norm, verbose=0)


oinv preproc: {'channel_order': ['acc_mag', 'gyro_mag', 'a_v', 'a_h', 'jerk_v', 'pitch_unwrapped'], 'static_channel_indices': [5], 'dynamic_channel_indices': [0, 1, 2, 3, 4], 'window_size': 128, 'step': 64, 'fs_hz': 50.0, 'feature_module': 'utils.orientation_invariant_features.compute_features'}


## Section 4 — Run both models on every labelled session

For each session we report:
- majority vote (the application-style "what activity is this?" call),
- per-window distribution of predictions (richer diagnostic — e.g. a
  walking session that the 12-channel model splits 70 % walking / 30 %
  standing is more telling than the majority label alone).

In [5]:
rows = []

for session, row in labels_df.iterrows():
    gt = int(row['activity_id'])
    df_raw = load_session(session)
    win12 = window_into_batches(df_raw.to_numpy(), 128, 64)
    if len(win12) == 0:
        print(f'WARN: session {session} has too few samples; skipping')
        continue
    win12_n = normalize_12ch(win12)
    probs12 = predict_12ch(win12_n)
    pred12 = probs12.argmax(axis=1)

    feats = compute_features(df_raw, fs_hz=50.0, group_cols=None, keep_meta=False)
    winN = window_into_batches(feats[ORIENTATION_INVARIANT_COLS].to_numpy(), 128, 64)
    winN_n = normalize_oinv(winN)
    probsN = predict_oinv(winN_n)
    predN = probsN.argmax(axis=1)

    maj12 = int(np.bincount(pred12, minlength=6).argmax())
    majN  = int(np.bincount(predN, minlength=6).argmax())
    correct12 = float((pred12 == gt).mean())
    correctN  = float((predN  == gt).mean())

    rows.append({
        'session': session,
        'orientation': row['pocket_orientation'],
        'true': ACT_LABELS[gt],
        '12ch_majority': ACT_LABELS[maj12],
        '12ch_correct_frac': correct12,
        '6ch_majority': ACT_LABELS[majN],
        '6ch_correct_frac': correctN,
        '6ch_better': correctN > correct12,
        'n_windows': len(pred12),
    })
    print(f'{session:<25s} gt={ACT_LABELS[gt]:<4s}  '
          f'12ch maj={ACT_LABELS[maj12]:<4s} ({correct12*100:5.1f}% windows) | '
          f'6ch maj={ACT_LABELS[majN]:<4s}  ({correctN*100:5.1f}% windows)')

res = pd.DataFrame(rows).set_index('session')


dws                       gt=dws   12ch maj=ups  (  0.0% windows) | 6ch maj=ups   (  0.0% windows)
ups                       gt=ups   12ch maj=ups  (100.0% windows) | 6ch maj=ups   (100.0% windows)
hod                       gt=wlk   12ch maj=dws  ( 22.2% windows) | 6ch maj=ups   (  0.0% windows)
hod2                      gt=wlk   12ch maj=wlk  (100.0% windows) | 6ch maj=wlk   (100.0% windows)
hodanje                   gt=wlk   12ch maj=ups  ( 38.5% windows) | 6ch maj=dws   ( 23.1% windows)
hodanje2                  gt=wlk   12ch maj=dws  (  0.0% windows) | 6ch maj=ups   (  0.0% windows)
hodanje3                  gt=wlk   12ch maj=ups  ( 14.3% windows) | 6ch maj=wlk   ( 65.7% windows)
jog                       gt=jog   12ch maj=ups  (  0.0% windows) | 6ch maj=ups   (  0.0% windows)
ED                        gt=wlk   12ch maj=wlk  ( 57.6% windows) | 6ch maj=wlk   ( 93.9% windows)
EG                        gt=wlk   12ch maj=wlk  (100.0% windows) | 6ch maj=ups   (  0.0% windows)
KD        

## Section 5 — Summary and decision

Two summary numbers per model:

- **Window-level accuracy across all sessions** — fraction of all
  windows (pooled across sessions) whose predicted class equals the
  session's ground-truth label. This is the cleanest robustness
  metric.
- **Session-level accuracy** — fraction of sessions where the majority
  vote equals the ground-truth label. This is what the eventual Flutter
  app will display, so it is the user-perceived metric.

Decision rule:

- If the 6-channel CNN's window-level accuracy is **strictly above**
  the 12-channel CNN's, this check passes and the orientation-invariant
  input becomes the input of record for the architecture step.
- If they are within ±1 pp, treat as a draw (no harm done) and still
  proceed to the architecture step because the architectural changes are what is
  expected to push robustness in the bigger margin.
- If 6-channel is clearly worse (> 1 pp below), revisit feature design
  (walking-direction frame, spectral features) before
  proceeding.

In [6]:
def aggregate(res_df: pd.DataFrame, key: str) -> tuple[float, float]:
    weights = res_df['n_windows'].to_numpy()
    fracs = res_df[key].to_numpy()
    window_level = float(np.average(fracs, weights=weights))
    session_level = float((fracs > 0.5).mean())  # session counts as correct if majority is right
    return window_level, session_level


win12_acc, sess12_acc = aggregate(res, '12ch_correct_frac')
winN_acc,  sessN_acc  = aggregate(res, '6ch_correct_frac')

summary = pd.DataFrame({
    'window_acc':  [win12_acc, winN_acc],
    'session_acc': [sess12_acc, sessN_acc],
}, index=['CNN-1D baseline (12ch raw)',
          'CNN-1D baseline (6ch orientation-invariant)'])
print(summary.round(4).to_string())

delta = winN_acc - win12_acc
print(f'\nΔ window-acc (new − old): {delta:+.4f}')
status = ('PASS — strict improvement' if delta > 0.01
          else ('DRAW — within ±1 pp' if abs(delta) <= 0.01
                else 'FAIL — clear regression'))
print(f'Acceptance: {status}')

os.makedirs('../results', exist_ok=True)
res.to_csv('../results/in_the_wild_oinv_per_session.csv')
summary.to_csv('../results/in_the_wild_oinv_summary.csv')
print('saved → ml/results/in_the_wild_oinv_*.csv')


                                             window_acc  session_acc
CNN-1D baseline (12ch raw)                       0.4667       0.4167
CNN-1D baseline (6ch orientation-invariant)      0.4042       0.4167

Δ window-acc (new − old): -0.0625
Acceptance: FAIL — clear regression
saved → ml/results/in_the_wild_oinv_*.csv


## Section 6 — Per-orientation breakdown

Walking sessions ED/EG/KD/KG carry the same ground truth (`wlk`) but
different pocket orientations. The whole point of the
orientation-invariant feature set is that those four sessions should
get *consistent* predictions. The table below makes the dependence
on orientation visible.

In [7]:
walk_only = res[res['true'] == 'wlk'].copy()
walk_only[['orientation', '12ch_correct_frac', '6ch_correct_frac', 'n_windows']]


,orientation,12ch_correct_frac,6ch_correct_frac,n_windows
session,,,,
hod,natural,0.222222,0.000000,9
hod2,natural,1.000000,1.000000,3
hodanje,natural,0.384615,0.230769,13
hodanje2,natural,0.000000,0.000000,13
hodanje3,natural,0.142857,0.657143,35
ED,screen-toward-thigh + upside-down,0.575758,0.939394,33
EG,screen-toward-thigh + upright,1.000000,0.000000,43
KD,camera-toward-thigh + upside-down,0.000000,0.000000,36
KG,camera-toward-thigh + upright,0.769231,0.820513,39


## Section 7 — Interpretation

The 6-channel orientation-invariant CNN regresses by ~6 pp window-accuracy relative to the 12-channel baseline (0.404 vs 0.467), so the hypothesis as stated — *any* positive lift on Android — **fails**. Session-level accuracy is identical (0.417), which is the weaker of the two metrics here because the per-window distribution is what reveals robustness.

The more useful signal is the per-orientation breakdown of the four ED/EG/KD/KG walking sessions, which carry the *same* gait but different pocket orientations. The whole point of orientation-invariant features is that those four should receive *consistent* predictions, even if uniformly wrong. Instead the 6-channel model gives 0.00 / 0.00 / 0.82 / 0.94 — a wider spread than the 12-channel baseline (0.58 / 1.00 / 0.00 / 0.77). So the feature set is not behaving as orientation-invariant in practice.

Two structural reasons for the failure:

1. **`pitch_unwrapped` is not orientation-invariant by construction.** It encodes the phone's tilt and changes sign / wrap point when the device is flipped (upside-down vs upright) or rotated 180° about the long axis (screen-toward-thigh vs camera-toward-thigh). Keeping it as a static channel re-injects exactly the orientation signal the rest of the feature set was designed to remove.
2. **`a_v`, `a_h`, `jerk_v` are invariant only conditional on a well-estimated gravity vector.** Android and iOS use different sensor-fusion pipelines for gravity (Stisen et al., 2015, *SenSys*), so the gravity estimate — and therefore the gravity-projected channels — differ between platforms even for the same physical motion. The on-MotionSense training distribution of `a_v / a_h` is not the on-Android inference distribution.

The natural next step, already prepared in notebook 08, is to move from a gravity-aligned frame to a **walking-direction frame** that aligns the horizontal axes with the direction of progression. This removes the residual yaw degree of freedom that `a_h` currently exposes to the classifier, and lets us drop `pitch_unwrapped` in favour of a heading-relative tilt that does not change sign under the ED/EG/KD/KG flips.